# FAME — DRVM temporal robustness and reference-risk audit (V12)

This notebook does **not introduce a new detector**. It stress-tests the same DRVM used in the manuscript and adds two targeted diagnostics for the remaining methodological risk:

1. **real-data reference informativeness and baseline-misspecification sensitivity**, using the already archived V10 deployment trajectories;
2. **temporal Monte Carlo stress tests** showing what happens when the bounded DRR score has predictable heteroskedasticity or a seasonal conditional mean.

The contextual extension uses the same DRVM recursion with a predictable threshold \(c_t\). For \(Q\) pre-specified calendar strata, each reference stratum receives error budget \(\gamma/Q\), so simultaneous reference coverage remains at least \(1-\gamma\) by the union bound.


In [1]:
from __future__ import annotations

import math
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/mnt/data")
RESULTS_ZIP = ROOT / "results_drvm_real_v10.zip"
OUT = ROOT / "results_drvm_temporal_robustness_v12"
OUT.mkdir(parents=True, exist_ok=True)

ALPHA = 0.04
GAMMA = 0.01
LAMBDAS = np.array([0.10, 0.25, 0.50, 1.00, 2.00, 4.00], dtype=float)
WEIGHTS = np.repeat(1.0/len(LAMBDAS), len(LAMBDAS))
PSI = np.exp(LAMBDAS) - 1.0 - LAMBDAS
THRESHOLD = 1.0 / ALPHA

N_MC = 1500
SEED = 20260910

print("Results:", RESULTS_ZIP)
print("Output:", OUT)


Results: /mnt/data/results_drvm_real_v10.zip
Output: /mnt/data/results_drvm_temporal_robustness_v12


## 1. Core DRVM functions

In [2]:
def binary_kl(p, q):
    eps = 1e-15
    p = min(max(float(p), eps), 1.0-eps)
    q = min(max(float(q), eps), 1.0-eps)
    return p*math.log(p/q) + (1-p)*math.log((1-p)/(1-q))


def kl_upper_mean(xbar, m, gamma):
    xbar = float(np.clip(xbar, 0.0, 1.0))
    if xbar >= 1.0:
        return 1.0

    target = math.log(1.0/gamma)
    lo, hi = xbar, 1.0 - 1e-15

    for _ in range(48):
        mid = (lo + hi) / 2.0
        if m * binary_kl(xbar, mid) > target:
            hi = mid
        else:
            lo = mid

    return (lo + hi) / 2.0


def run_drvm_predictable(x, c_t, alpha=ALPHA):
    x = np.asarray(x, dtype=float)
    c_t = np.asarray(c_t, dtype=float)

    A = np.zeros(len(LAMBDAS), dtype=float)
    tau = None
    rows = []

    for idx, (xt, c) in enumerate(zip(x, c_t)):
        t = idx + 1
        rho_t = 1.0 / (t*(t+1.0))

        if c >= 1.0:
            factor = np.zeros(len(LAMBDAS))
        else:
            v = c*(1-c) if c <= 0.5 else 0.25
            factor = np.exp(
                np.clip(
                    LAMBDAS*(xt-c) - PSI*v,
                    -745.0, 700.0
                )
            )

        A = factor * (A + rho_t)
        C_t = 1.0/(t+1.0) + float(A @ WEIGHTS)

        rows.append((t, xt, c, C_t))

        if tau is None and C_t >= 1.0/alpha:
            tau = t

    traj = pd.DataFrame(rows, columns=["t","x","c_t","C_t"])

    return {
        "tau": tau,
        "max_C_t": float(traj["C_t"].max()),
        "final_C_t": float(traj["C_t"].iloc[-1]),
        "trajectory": traj,
    }


def scalar_drvm(x, c, alpha=ALPHA):
    return run_drvm_predictable(
        x,
        np.repeat(float(c), len(x)),
        alpha=alpha
    )


## 2. Load V10 real-data results

In [3]:
with zipfile.ZipFile(RESULTS_ZIP) as z:
    cartola_summary = pd.read_csv(z.open("cartola_drvm_summary_v10.csv"))
    cartola_traj = pd.read_csv(z.open("cartola_drvm_trajectory_v10.csv"))
    energy_summary = pd.read_csv(z.open("energy_drvm_summary_v10.csv"))
    energy_traj = pd.read_csv(z.open("energy_drvm_trajectories_v10.csv"))

print("Cartola episodes:", len(cartola_summary))
print("Energy episodes:", len(energy_summary))


Cartola episodes: 1
Energy episodes: 24


## 3. Reference-informativeness audit

The quantity

\[
b_R=c-\bar X_R
\]

is the empirical reference buffer produced by the upper reference bound. We compare it with the observed deployment shift

\[
s_E=\bar X_E-\bar X_R.
\]

The ratio \(s_E/b_R\) is **descriptive**, not a test statistic. Values above one mean that the average deployment shift is larger than the empirical reference buffer.


In [4]:
energy_audit = energy_summary.copy()
energy_audit["domain"] = "Energy"
energy_audit["episode"] = (
    energy_audit["model"].astype(str)
    + "-"
    + energy_audit["deployment_year"].astype(str)
)
energy_audit["reference_buffer"] = (
    energy_audit["c"] - energy_audit["mean_x_reference"]
)
energy_audit["deployment_shift"] = (
    energy_audit["mean_x_deployment"] - energy_audit["mean_x_reference"]
)
energy_audit["shift_buffer_ratio"] = (
    energy_audit["deployment_shift"] / energy_audit["reference_buffer"]
)

cartola_audit = pd.DataFrame([{
    "domain": "Fantasy football",
    "episode": "2025",
    "model": "Cartola",
    "deployment_year": 2025,
    "mean_x_reference": float(cartola_summary["mean_x_reference"].iloc[0]),
    "mean_x_deployment": float(cartola_summary["mean_x_deployment"].iloc[0]),
    "c": float(cartola_summary["c"].iloc[0]),
    "status": str(cartola_summary["status"].iloc[0]),
}])
cartola_audit["reference_buffer"] = (
    cartola_audit["c"] - cartola_audit["mean_x_reference"]
)
cartola_audit["deployment_shift"] = (
    cartola_audit["mean_x_deployment"] - cartola_audit["mean_x_reference"]
)
cartola_audit["shift_buffer_ratio"] = (
    cartola_audit["deployment_shift"] / cartola_audit["reference_buffer"]
)

audit = pd.concat([
    energy_audit[[
        "domain","episode","model","deployment_year",
        "mean_x_reference","mean_x_deployment","c","status",
        "reference_buffer","deployment_shift","shift_buffer_ratio"
    ]],
    cartola_audit
], ignore_index=True)

audit.to_csv(OUT / "reference_informativeness_audit_v12.csv", index=False)

display(
    audit.sort_values("shift_buffer_ratio", ascending=False).head(10)
)


,domain,episode,model,deployment_year,mean_x_reference,mean_x_deployment,c,status,reference_buffer,deployment_shift,shift_buffer_ratio
9,Energy,ENTSOE-2021,ENTSOE,2021,0.283266,0.475087,0.357951,signal,0.074685,0.191821,2.568413
19,Energy,SARIMAX-2024,SARIMAX,2024,0.129146,0.261095,0.188450,signal,0.059304,0.131948,2.224956
13,Energy,SARIMAX-2022,SARIMAX,2022,0.160701,0.300185,0.224473,signal,0.063772,0.139484,2.187247
11,Energy,LSTM-2021,LSTM,2021,0.117381,0.157842,0.174684,no_signal,0.057303,0.040461,0.706093
21,Energy,ENTSOE-2025,ENTSOE,2025,0.099768,0.132523,0.153898,no_signal,0.054131,0.032756,0.605119
7,Energy,SARIMAX-2020,SARIMAX,2020,0.283512,0.324958,0.358317,no_signal,0.074805,0.041446,0.554050
18,Energy,ENTSOE-2024,ENTSOE,2024,0.082341,0.107045,0.132969,no_signal,0.050629,0.024704,0.487953
20,Energy,LSTM-2024,LSTM,2024,0.116403,0.143708,0.173627,no_signal,0.057223,0.027304,0.477155
1,Energy,SARIMAX-2018,SARIMAX,2018,0.252561,0.277249,0.325338,no_signal,0.072777,0.024688,0.339233
5,Energy,LSTM-2019,LSTM,2019,0.117515,0.136744,0.174927,no_signal,0.057412,0.019229,0.334933


## 4. Adversarial baseline-uplift sensitivity

To assess how much upward misspecification of the scalar baseline each real signal can tolerate, rerun the **same** DRVM after replacing

\[
c \quad\text{by}\quad c+\varepsilon,
\]

with no other change. Because each DRVM log increment decreases as \(c\) increases, a constant uplift \(\varepsilon\) is conservative relative to any nonnegative predictable baseline perturbation bounded by \(\varepsilon\).

For each signaling episode we calculate the largest \(\varepsilon\) for which the signal is still produced.


In [5]:
def critical_uplift(x, c):
    base = scalar_drvm(x, c)
    if base["tau"] is None:
        return 0.0

    lo, hi = 0.0, max(0.0, 1.0-c)

    for _ in range(55):
        mid = (lo + hi) / 2.0
        res = scalar_drvm(x, c + mid)
        if res["tau"] is not None:
            lo = mid
        else:
            hi = mid

    return lo


uplift_grid = [0.00, 0.005, 0.010, 0.025, 0.050, 0.075, 0.100]
sens_rows = []
critical_rows = []

for _, row in energy_summary[energy_summary["status"].eq("signal")].iterrows():
    g = energy_traj[
        (energy_traj["deployment_year"].eq(row["deployment_year"]))
        & (energy_traj["model"].eq(row["model"]))
    ].sort_values("t")

    x = g["x"].to_numpy()
    c = float(row["c"])
    crit = critical_uplift(x, c)

    critical_rows.append({
        "domain": "Energy",
        "episode": f"{row['model']}-{int(row['deployment_year'])}",
        "model": row["model"],
        "deployment_year": int(row["deployment_year"]),
        "c_original": c,
        "critical_uplift": crit,
        "critical_uplift_pct_of_c": 100.0*crit/c,
    })

    for eps in uplift_grid:
        res = scalar_drvm(x, c+eps)
        sens_rows.append({
            "domain": "Energy",
            "episode": f"{row['model']}-{int(row['deployment_year'])}",
            "epsilon": eps,
            "c_sensitivity": c+eps,
            "signal": int(res["tau"] is not None),
            "time_to_signal": res["tau"],
            "max_C_t": res["max_C_t"],
            "final_C_t": res["final_C_t"],
        })

critical = pd.DataFrame(critical_rows)
sensitivity = pd.DataFrame(sens_rows)

critical.to_csv(OUT / "critical_baseline_uplift_v12.csv", index=False)
sensitivity.to_csv(OUT / "baseline_uplift_sensitivity_v12.csv", index=False)

display(critical)


,domain,episode,model,deployment_year,c_original,critical_uplift,critical_uplift_pct_of_c
0,Energy,ENTSOE-2021,ENTSOE,2021,0.357951,0.042561,11.890088
1,Energy,SARIMAX-2022,SARIMAX,2022,0.224473,0.007524,3.351912
2,Energy,SARIMAX-2024,SARIMAX,2024,0.188450,0.007011,3.720372


In [6]:
plt.figure(figsize=(8.4, 5.2))

for episode, g in sensitivity.groupby("episode"):
    g = g.sort_values("epsilon")
    plt.plot(
        g["epsilon"],
        g["max_C_t"],
        marker="o",
        label=episode
    )

plt.axhline(THRESHOLD, linestyle="--", label=r"$1/\alpha=25$")
plt.yscale("log")
plt.xlabel(r"Additional baseline uplift $\varepsilon$")
plt.ylabel(r"Maximum DRVM evidence $\max_t C_t$")
plt.legend(frameon=False)
plt.tight_layout()

plt.savefig(OUT / "fig_baseline_uplift_sensitivity_v12.pdf")
plt.savefig(OUT / "fig_baseline_uplift_sensitivity_v12.png", dpi=220)
plt.close()


## 5. Contextual-reference extension

For pre-specified predictable strata \(q_t\in\{1,\ldots,Q\}\), build a reference upper bound in each stratum using error budget \(\gamma/Q\) and set

\[
c_t=r^U_{0,q_t}+\delta_X.
\]

The DRVM recursion is unchanged except that \(c_t\) and its corresponding variance budget \(v_t\) are updated predictably over time.


In [7]:
def contextual_threshold_from_reference(x_ref, Q, gamma=GAMMA):
    x_ref = np.asarray(x_ref, dtype=float)
    n = len(x_ref)
    groups = np.floor(np.arange(n)*Q/n).astype(int)

    cvals = np.zeros(Q)

    for q in range(Q):
        vals = x_ref[groups == q]
        cvals[q] = kl_upper_mean(
            vals.mean(),
            len(vals),
            gamma/Q
        )

    c_t = np.array([cvals[q] for q in groups])
    return c_t, cvals


## 6. Temporal Monte Carlo stress test

Three stress regimes are used, all with 365 reference and 365 deployment observations.

**A. Predictable heteroskedastic null.** The conditional mean is constant at \(0.15\), while the amplitude of a bounded mean-zero innovation varies by quarter. This satisfies the scalar conditional-mean null despite non-identical conditional distributions.

**B. Seasonal no-deterioration regime.** The first semester has Bernoulli mean \(0.01\) and the second semester mean \(0.45\), identically in reference and deployment. A scalar annual baseline is deliberately misspecified because the valid conditional mean changes predictably over the year. Both the semiannual (\(Q=2\)) and quarterly (\(Q=4\)) contextual specifications are correctly specified; \(Q=4\) is intentionally finer than necessary.

**C. Seasonal deterioration.** The same semiannual pattern is used in reference, and deployment means are increased by \(0.15\), giving \(0.16\) and \(0.60\).

We compare \(Q=1\) (scalar), \(Q=2\) (correct semiannual contextual reference), and \(Q=4\) (correct but unnecessarily fine quarterly reference). This design cleanly separates protection against predictable mean heterogeneity from the loss of information caused by over-stratification.


In [8]:
N = 365
quarter = np.floor(np.arange(N)*4/N).astype(int)

# Semiannual seasonal mean: Q=2 is correctly specified; Q=4 is a finer,
# also correctly specified partition.
seasonal_means = np.array([0.01, 0.01, 0.45, 0.45])[quarter]
seasonal_shifted = np.clip(seasonal_means + 0.15, 0.0, 1.0)

hetero_amp = np.array([0.02, 0.10, 0.04, 0.08])[quarter]
HETERO_MU = 0.15


def generate_stress_pair(rng, scenario):
    if scenario == "heteroskedastic_null":
        x_ref = HETERO_MU + hetero_amp*rng.choice([-1.0,1.0], size=N)
        x_dep = HETERO_MU + hetero_amp*rng.choice([-1.0,1.0], size=N)
        return x_ref, x_dep

    if scenario == "seasonal_no_deterioration":
        x_ref = rng.binomial(1, seasonal_means).astype(float)
        x_dep = rng.binomial(1, seasonal_means).astype(float)
        return x_ref, x_dep

    if scenario == "seasonal_deterioration":
        x_ref = rng.binomial(1, seasonal_means).astype(float)
        x_dep = rng.binomial(1, seasonal_shifted).astype(float)
        return x_ref, x_dep

    raise ValueError(scenario)


scenarios = [
    "heteroskedastic_null",
    "seasonal_no_deterioration",
    "seasonal_deterioration",
]

raw_rows = []
seed_seq = np.random.SeedSequence(SEED)
children = seed_seq.spawn(len(scenarios)*3*N_MC)
k = 0

for scenario in scenarios:
    for Q in [1,2,4]:
        for rep in range(N_MC):
            rng = np.random.default_rng(children[k])
            k += 1

            x_ref, x_dep = generate_stress_pair(rng, scenario)
            c_t, cvals = contextual_threshold_from_reference(x_ref, Q)
            res = run_drvm_predictable(x_dep, c_t)

            raw_rows.append({
                "scenario": scenario,
                "Q": Q,
                "rep": rep,
                "signal": int(res["tau"] is not None),
                "time_to_signal": res["tau"],
                "max_C_t": res["max_C_t"],
                "mean_reference": x_ref.mean(),
                "mean_deployment": x_dep.mean(),
                "max_c_t": c_t.max(),
                "min_c_t": c_t.min(),
            })

stress_raw = pd.DataFrame(raw_rows)

stress_summary = (
    stress_raw.groupby(["scenario","Q"], as_index=False)
    .agg(
        n=("rep","size"),
        signal_probability=("signal","mean"),
        median_time_to_signal=("time_to_signal","median"),
        mean_reference=("mean_reference","mean"),
        mean_deployment=("mean_deployment","mean"),
        mean_min_c=("min_c_t","mean"),
        mean_max_c=("max_c_t","mean"),
    )
)

stress_raw.to_csv(OUT / "temporal_stress_raw_v12.csv", index=False)
stress_summary.to_csv(OUT / "temporal_stress_summary_v12.csv", index=False)

display(stress_summary)


,scenario,Q,n,signal_probability,median_time_to_signal,mean_reference,mean_deployment,mean_min_c,mean_max_c
0,heteroskedastic_null,1,1500,0.000000,NaN,0.150013,0.149957,0.212365,0.212365
1,heteroskedastic_null,2,1500,0.000000,NaN,0.149799,0.149932,0.245077,0.251652
2,heteroskedastic_null,4,1500,0.000000,NaN,0.150027,0.150028,0.297192,0.315595
3,seasonal_deterioration,1,1500,0.993333,263.0,0.229332,0.379671,0.300195,0.300195
4,seasonal_deterioration,2,1500,0.946667,55.0,0.229666,0.378411,0.053427,0.570386
5,seasonal_deterioration,4,1500,0.606000,84.0,0.229165,0.378590,0.076055,0.655542
6,seasonal_no_deterioration,1,1500,0.195333,334.0,0.229733,0.229943,0.300627,0.300627
7,seasonal_no_deterioration,2,1500,0.000000,NaN,0.229821,0.230303,0.052579,0.571082
8,seasonal_no_deterioration,4,1500,0.000000,NaN,0.230152,0.228727,0.076046,0.657882


In [9]:
plt.figure(figsize=(8.4, 5.2))

for scenario, g in stress_summary.groupby("scenario"):
    g = g.sort_values("Q")
    plt.plot(
        g["Q"],
        g["signal_probability"],
        marker="o",
        label=scenario.replace("_", " ")
    )

plt.xlabel("Number of pre-specified reference strata Q")
plt.ylabel("Empirical signal probability")
plt.xticks([1,2,4])
plt.ylim(-0.03, 1.03)
plt.legend(frameon=False)
plt.tight_layout()

plt.savefig(OUT / "fig_temporal_stress_v12.pdf")
plt.savefig(OUT / "fig_temporal_stress_v12.png", dpi=220)
plt.close()


## 7. Compact manuscript tables

In [10]:
compact_real = critical.copy()
compact_real["survives_eps_0_01"] = (
    compact_real["critical_uplift"] >= 0.01
)
compact_real["survives_eps_0_025"] = (
    compact_real["critical_uplift"] >= 0.025
)

compact_real.to_csv(
    OUT / "manuscript_real_sensitivity_v12.csv",
    index=False
)

compact_stress = stress_summary.copy()
compact_stress.to_csv(
    OUT / "manuscript_temporal_stress_v12.csv",
    index=False
)

display(compact_real)
display(compact_stress)


,domain,episode,model,deployment_year,c_original,critical_uplift,critical_uplift_pct_of_c,survives_eps_0_01,survives_eps_0_025
0,Energy,ENTSOE-2021,ENTSOE,2021,0.357951,0.042561,11.890088,True,True
1,Energy,SARIMAX-2022,SARIMAX,2022,0.224473,0.007524,3.351912,False,False
2,Energy,SARIMAX-2024,SARIMAX,2024,0.188450,0.007011,3.720372,False,False


,scenario,Q,n,signal_probability,median_time_to_signal,mean_reference,mean_deployment,mean_min_c,mean_max_c
0,heteroskedastic_null,1,1500,0.000000,NaN,0.150013,0.149957,0.212365,0.212365
1,heteroskedastic_null,2,1500,0.000000,NaN,0.149799,0.149932,0.245077,0.251652
2,heteroskedastic_null,4,1500,0.000000,NaN,0.150027,0.150028,0.297192,0.315595
3,seasonal_deterioration,1,1500,0.993333,263.0,0.229332,0.379671,0.300195,0.300195
4,seasonal_deterioration,2,1500,0.946667,55.0,0.229666,0.378411,0.053427,0.570386
5,seasonal_deterioration,4,1500,0.606000,84.0,0.229165,0.378590,0.076055,0.655542
6,seasonal_no_deterioration,1,1500,0.195333,334.0,0.229733,0.229943,0.300627,0.300627
7,seasonal_no_deterioration,2,1500,0.000000,NaN,0.229821,0.230303,0.052579,0.571082
8,seasonal_no_deterioration,4,1500,0.000000,NaN,0.230152,0.228727,0.076046,0.657882


## 8. Save configuration and archive

In [11]:
pd.DataFrame([{
    "alpha": ALPHA,
    "gamma": GAMMA,
    "lambdas": str(LAMBDAS.tolist()),
    "n_mc_per_scenario_Q": N_MC,
    "stress_horizon": N,
    "seasonal_quarter_means": str([0.01,0.01,0.45,0.45]),
    "seasonal_deterioration": 0.15,
    "heteroskedastic_null_mean": HETERO_MU,
    "heteroskedastic_quarter_amplitudes": str([0.02,0.10,0.04,0.08]),
}]).to_csv(OUT / "config_v12.csv", index=False)

archive = shutil.make_archive(
    "/mnt/data/results_drvm_temporal_robustness_v12",
    "zip",
    root_dir=OUT
)
print("Archive:", archive)


Archive: /mnt/data/results_drvm_temporal_robustness_v12.zip
